# পাঠ ০৮ - মাল্টি-এজেন্ট ডিজাইন প্যাটার্ন


## সেটআপ


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## কেন মাল্টি-এজেন্ট সিস্টেম?

বাস্তব জীবন টাস্ক যেমন ট্রিপ প্ল্যানিং অনেক ধরনের দক্ষতার প্রয়োজন হয় — লজিস্টিকস, স্থানীয় জ্ঞান, বাজেট নির্মাণ, এবং আরও অনেক কিছু। একটি একক এজেন্ট সবকিছু সামলানোর চেষ্টা করলে দ্রুত অপ্রতুল এবং অপ্রশস্ত হয়ে ওঠে।

মাল্টি-এজেন্ট সিস্টেম এটি সমাধান করে **বিশেষীকরণের** মাধ্যমে: প্রতিটি এজেন্ট একটি বিশেষ দক্ষতার ক্ষেত্রে মনোনিবেশ করে, যা সাধারণজ্ঞের তুলনায় উচ্চগুণমানের ফলাফল প্রদান করে। এরা **স্কেলেবিলিটি** ও উন্নত করে — আপনি নতুন এজেন্ট যোগ করতে পারেন (যেমন, একটি উড়ান বিশেষজ্ঞ, একটি রেস্টুরেন্ট সমালোচক) বিদ্যমান ওয়ার্কফ্লো পুনরায় লেখার প্রয়োজন ছাড়াই। এজেন্টগুলি একটি গঠনমূলক পাইপলাইন মাধ্যমে একসাথে সংযুক্ত হয়, এক থেকে অন্যের কাছে প্রসঙ্গ পরিবহন করে।


## বিশেষায়িত এজেন্ট তৈরি করা


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ধারাবাহিক ওয়ার্কফ্লো তৈরি করা

`WorkflowBuilder` আপনাকে এজেন্টদের একটি নির্দেশিত গ্রাফে সংযুক্ত করতে দেয়। এখানে আমরা একটি সহজ দুই ধাপের পাইপলাইন তৈরি করি: **TravelPlanner** যাত্রার তালিকা প্রস্তুত করে, তারপর **TravelConcierge** এটি পর্যালোচনা করে এবং উন্নত করে।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ওয়ার্কফ্লোতে আরও এজেন্ট যুক্ত করা

মাল্টি-এজেন্ট প্যাটার্নের সবচেয়ে বড় সুবিধাগুলোর একটি হলো এটি সহজেই বাড়ানো যায়। নিচে আমরা একটি **BudgetReviewer** এজেন্ট যোগ করেছি যা ভ্রমণকারীর বাজেটের সাথে পরিকল্পনাটি পরীক্ষা করে, এমন আইটেম নির্দেশ করে যেগুলো খরচ সীমার বাইরে যেতে পারে, এবং অর্থ সাশ্রয়ের বিকল্পগুলি প্রস্তাব করে। ওয়ার্কফ্লো এখন তিনটি এজেন্ট ধারাবাহিকভাবে চালায়:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## সারসংক্ষেপ

এই পাঠে আপনি শিখেছেন কিভাবে:

1. **বিশেষায়িত এজেন্ট তৈরি করবেন** — প্রত্যেকের একটি নির্দিষ্ট ভূমিকা থাকবে (পরিকল্পনা, কনসিয়ার্জ, বাজেট পর্যালোচনা)।
2. **এজেন্টগুলোকে ধারাবাহিক ওয়ার্কফ্লোতে সংযুক্ত করবেন** `WorkflowBuilder` এবং `add_edge` ব্যবহার করে।
3. **মাল্টি-এজেন্ট পাইপলাইন থেকে আউটপুট স্ট্রিম করবেন**, কোন এজেন্ট কথা বলছে তা ট্র্যাক করে।
4. **ওয়ার্কফ্লো সম্প্রসারণ করবেন** নতুন এজেন্ট যোগ করে, বিদ্যমানগুলো পরিবর্তন না করেই।

মাল্টি-এজেন্ট ডিজাইন প্যাটার্ন প্রত্যেক এজেন্টকে সরল রাখে যখন একক এজেন্টের থেকে অনেক বেশি সমৃদ্ধ ও সুসমীক্ষিত ফলাফল উৎপাদন করে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকারোক্তি**:  
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। আমরা যথাসাধ্য সঠিকতা নিশ্চিত করার চেষ্টা করি, তবে স্বয়ংক্রিয় অনুবাদে ভুল বা অসঙ্গতি থাকতে পারে। মূল ভাষায় থাকা নথিটিকেই কর্তৃত্বপ্রাপ্ত উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের ক্ষেত্রে পেশাদার মানব অনুবাদ গ্রহণ করার পরামর্শ দেওয়া হয়। এই অনুবাদের ব্যবহারের ফলে কোনো ভুলবোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নয়।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
